# Generating Brazilian Names with Bigram Models

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from matplotlib.colors import PowerNorm

from bznames.ibge import load_ibge_name_data
from bznames.metrics import (
    compute_bigram_nll_for_tokens,
)
from bznames.tokenizer import CharacterEncoder, tokenize_dataset
from bznames.viz import (
    display_bigram_model_nll_comparison,
    display_bigram_model_samples,
    display_tokenized_examples,
)

%config InlineBackend.figure_format = 'retina'

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### Data Loading

In [ ]:
data = load_ibge_name_data()

len(data)

#### Simple Data Exploration

In [ ]:
data[:5]

In [ ]:
np.random.seed(42)
np.random.choice(data, size=5, replace=False).tolist()

In [ ]:
print(
    "Total number of people represented in the dataset:",
    sum(x["freq"] for x in data),
)

### Frequentist Bigram Model

In [ ]:
SPECIAL_TOKEN = "."

names = [x["name"] for x in data]
encoder = CharacterEncoder.from_words(names, special_token=SPECIAL_TOKEN)
vocab_size = encoder.vocab_size

In [ ]:
ngram_size = 2

input_tokens, output_tokens, freqs = tokenize_dataset(
    data,
    encoder,
    ngram_size=ngram_size,
    show_progress=True,
)

In [ ]:
display_tokenized_examples(
    input_tokens,
    output_tokens,
    freqs,
    encoder,
    limit=10,
    format_type="html",  # Options: "text", "markdown", "html"
)


In [ ]:
input_tokens = torch.tensor(input_tokens, dtype=torch.int32)
output_tokens = torch.tensor(output_tokens, dtype=torch.int32)
freqs = torch.tensor(freqs, dtype=torch.int32)
sample_weights = freqs / freqs.sum()

In [ ]:
counts = torch.zeros((vocab_size, vocab_size), dtype=torch.int32)

# Extract row and column indices
row_indices = input_tokens[:, 0]
col_indices = output_tokens

# Perform vectorized in-place addition with accumulation
counts.index_put_((row_indices, col_indices), freqs, accumulate=True)

# Laplace smoothing
counts += 1

In [ ]:
uniform_probs = torch.ones_like(counts, dtype=torch.float32)
uniform_probs /= uniform_probs.sum(dim=1, keepdim=True)

bigram_uncond_probs = counts / counts.sum()
bigram_cond_probs = counts / counts.sum(dim=1, keepdim=True)

In [ ]:
# Plot the unconditional bigram distribution
fig, ax = plt.subplots(dpi=200, figsize=(5, 5))

ax.imshow(bigram_uncond_probs, cmap="Blues", norm=PowerNorm(gamma=0.5))
ax.set_xticks([])
ax.set_yticks([])

for i in range(vocab_size):
    for j in range(vocab_size):
        bigram = encoder.decode_index(i) + encoder.decode_index(j)
        ax.text(j, i, bigram, ha="center", va="bottom", color="gray", size=5)
        ax.text(
            x=j,
            y=i,
            s=f"{100 * bigram_uncond_probs[i, j].item():.2f}%",
            ha="center",
            va="top",
            color="gray",
            size=3,
        )

ax.set_title("Unconditional Bigram Distribution", size=8);

In [ ]:
# Plot the conditional bigram distributions, each row represents a distribution
fig, ax = plt.subplots(dpi=200, figsize=(5, 5))

ax.imshow(bigram_cond_probs, cmap="Blues", norm=PowerNorm(gamma=0.5))
ax.set_xticks([])
ax.set_yticks([])

for i in range(vocab_size):
    for j in range(vocab_size):
        bigram = encoder.decode_index(i) + encoder.decode_index(j)

        ax.text(j, i, bigram, ha="center", va="bottom", color="gray", size=5)
        ax.text(
            x=j,
            y=i,
            s=f"{100 * bigram_cond_probs[i, j].item():.2f}%",
            ha="center",
            va="top",
            color="gray",
            size=2.5,
        )

ax.set_title("Conditional Bigram Distributions", size=8);

In [ ]:
display_bigram_model_samples(
    {
        "Computed Bigram Model": bigram_cond_probs,
        "Uniform Bigram Model": uniform_probs,
    },
    encoder=encoder,
    num_samples=10,
)


In [ ]:
bigram_nll = compute_bigram_nll_for_tokens(
    bigram_cond_probs,
    input_tokens,
    output_tokens,
    sample_weights,
)
uniform_nll = compute_bigram_nll_for_tokens(
    uniform_probs,
    input_tokens,
    output_tokens,
    sample_weights,
)

display_bigram_model_nll_comparison(
    models={
        "Computed Bigram Model": {
            "probs": bigram_cond_probs,
            "dataset_nll": bigram_nll,
        },
        "Uniform Bigram Model": {
            "probs": uniform_probs,
            "dataset_nll": uniform_nll,
        },
    },
    encoder=encoder,
    test_names=["rafael", "gabriela", "joao", "maria", "qkja", "kppsf"],
)

### Training a Neural Network to learn the Bigram Model

In [ ]:
# Create one-hot encodings for the input and output
X = F.one_hot(input_tokens.squeeze(), num_classes=vocab_size).float()
y = F.one_hot(output_tokens, num_classes=vocab_size).float()

In [ ]:
# Compute the negative log-likelihood for the entire dataset, using the neural network weights
def nn_compute_nll_for_dataset(
    X: torch.Tensor,
    W: torch.Tensor,
    next_idxs: torch.Tensor,
    weights: torch.Tensor,
    epsilon: float = 1e-7,
) -> float:
    assert X.shape[0] == len(next_idxs) == len(weights)

    # Forward pass
    logits = X @ W
    nn_counts = logits.exp()
    nn_probs = nn_counts / nn_counts.sum(dim=1, keepdim=True)

    rows = torch.arange(len(next_idxs))
    target_probs = nn_probs[rows, next_idxs]

    # sums epsilon to avoid log(0)
    nll = -1 * (weights * torch.log(target_probs + epsilon)).sum()

    return nll


# Compute the negative log-likelihood for a given name, using the neural network weights
def nn_compute_nll_for_name(name: str, W: torch.Tensor) -> float:
    name = "." + name + "."

    idxs = torch.tensor([encoder.encode_char(c) for c in name[:-1]])
    next_idxs = torch.tensor([encoder.encode_char(c) for c in name[1:]])

    # one-hot encoding of the input
    X = F.one_hot(input_tokens.squeeze(), num_classes=vocab_size).float()

    # Forward pass
    logits = X @ W
    counts = logits.exp()
    nn_probs = counts / counts.sum(dim=1, keepdim=True)

    # Get the probabilities for the actual next characters
    rows = torch.arange(len(next_idxs))
    target_probs = nn_probs[rows, next_idxs]

    nll = -1 * torch.log(target_probs).mean()

    return nll.item()

In [ ]:
print("------------- Untrained Neural Network with Zero Weights -------------")
W = torch.zeros((vocab_size, vocab_size))
zero_nn_nll = nn_compute_nll_for_dataset(X, W, output_tokens, sample_weights)

print(f"\nNegative Log-Likelihood for the dataset: \n{zero_nn_nll.item():.3f}")

print("\nNegative log-likelihoods for some real names:")
for name in ["rafael", "gabriela", "joao", "maria", "qkja", "kppsf"]:
    nll = nn_compute_nll_for_name(name, W)
    print(f" * '{name}': {nll:.3f}")

In [ ]:
(vocab_size - 1) ** 10

In [ ]:
print("------------- Untrained Neural Network with Normal Weights -------------")
g = torch.Generator().manual_seed(42)
W = torch.randn((vocab_size, vocab_size), generator=g, requires_grad=True)

loss = nn_compute_nll_for_dataset(X, W, output_tokens, sample_weights)
print(f"\nNegative Log-Likelihood for the dataset: \n{loss.item():.3f}")

print("\nNegative log-likelihoods for some real names:")
for name in ["rafael", "gabriela", "joao", "maria", "qkja", "kppsf"]:
    nll = nn_compute_nll_for_name(name, W)
    print(f" * '{name}': {nll:.3f}")

In [ ]:
# Checking the equivalence between the custom negative log-likelihood function
# and an alternative implementation using PyTorch's built-in cross-entropy function.
logits = X @ W
entropy_values = F.cross_entropy(X @ W, y, reduction="none")
weighted_cross_entropy = (sample_weights * entropy_values).sum()

torch.isclose(loss, weighted_cross_entropy).item()

In [ ]:
# Training the neural network using gradient descent
loss_values = [loss.item()]

for i in range(500 + 1):
    # Backpropagation
    W.grad = None
    loss.backward()

    # Weight update, using a learning rate of 50
    with torch.no_grad():
        W -= 50 * W.grad

    loss = nn_compute_nll_for_dataset(X, W, output_tokens, sample_weights)
    loss_values.append(loss.item())

    if i % 20 == 0:
        print(f"{i=} | loss={loss.item():.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

ax.plot(np.arange(len(loss_values)), loss_values)
ax.axhline(
    bigram_nll,
    color="red",
    linestyle="--",
    label="Computed bigram model",
)
ax.axhline(
    uniform_nll,
    color="purple",
    linestyle="--",
    label="Uniform bigram model",
)

ax.set_xlabel("epoch")
ax.set_ylabel("Loss value")
ax.set_title("Negative Log-Likelihood during neural net training")

ax.legend();

The Negative Log-Likelihood for the trained neural network using the bigrams as features and targets closely approaches that of the original bigram model, as the neural network effectively learns the probability distributions produced by the bigram model.

In [ ]:
print("------------- Trained Neural Network -------------")
loss = nn_compute_nll_for_dataset(X, W, output_tokens, sample_weights)
print(f"\nNegative Log-Likelihood for the dataset: \n{loss.item():.3f}")

print("\nNegative log-likelihoods for some real names:")
for name in ["rafael", "gabriela", "joao", "maria", "qkja", "kppsf"]:
    nll = nn_compute_nll_for_name(name, W)
    print(f" * '{name}': {nll:.3f}")

In [ ]:
def nn_sample_name(W):
    i = 0
    name = []

    while True:
        p = W[i].exp() / W[i].exp().sum()
        i = torch.multinomial(p, num_samples=1, generator=g).item()

        if i == 0:
            break
        else:
            name.append(encoder.decode_index(i))

    return "".join(name)


print("Sampling using the trained neural network:")
for _ in range(10):
    print(" - " + nn_sample_name(W))

print("\nSampling using a neural neural network with zero weights:")
for _ in range(10):
    print(" - " + nn_sample_name(torch.zeros_like(W)))

#### Checking the Learned Probabilities

In [ ]:
# The character '.' is used to represent the start and end of a name, with its index assigned as 0.
x0 = F.one_hot(torch.tensor([0]), num_classes=vocab_size).float()

# one-hot encoding of the 0 index:
x0

In [ ]:
# Logits for the first character
logits_x0 = x0 @ W

logits_x0

In [ ]:
# Matrix multiplication when X is a single one-hot encoded vector is equivalent to indexing a row of W.
torch.allclose(logits_x0, W[0])

In [ ]:
# The probabilities for the first character, after applying the softmax function to the linear logits.
nn_probs_x0 = logits_x0.exp() / logits_x0.exp().sum()

nn_probs_x0.detach().numpy().round(decimals=3)

In [ ]:
# Checking the above softmax calculation matches with the PyTorch implementation.
torch.allclose(nn_probs_x0, torch.softmax(logits_x0, dim=1))

In [ ]:
def cosine_similarity(a: torch.Tensor, b: torch.Tensor) -> float:
    return (a.dot(b) / (a.norm() * b.norm())).item()


# Cosine similarity between the probability vectors for the first character,
# with the left side computed using the neural network and
# the right side using the bigram model.
cosine_similarity(nn_probs_x0.flatten(), bigram_cond_probs[0])

In [ ]:
print(
    "Cosine similarities between learned and bigram probability vectors, for each starting character:"
)
for i in range(vocab_size):
    x_enc = F.one_hot(torch.tensor([i]), num_classes=vocab_size).float()
    logits_i = x_enc @ W
    nn_probs_i = logits_i.exp() / logits_i.exp().sum()

    print(
        f" '{encoder.decode_index(i)}': {cosine_similarity(nn_probs_i.flatten(), bigram_cond_probs[i]):.3f}"
    )

#### Visualizing the Learned Probabilities

In [ ]:
# Cosine similarity between the flattened matrices of probabilities
all_idxs = torch.arange(vocab_size)
x_enc = F.one_hot(all_idxs, num_classes=vocab_size).float()
logits_all = x_enc @ W
nn_probs_all = logits_all.exp() / logits_all.exp().sum(dim=1, keepdim=True)

cosine_similarity(nn_probs_all.flatten(), bigram_cond_probs.flatten())

Very high cosine similarity between the two probability vectors shows again that the neural network has learned the bigram model.

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(16, 16))

ax[0].imshow(nn_probs_all.detach().numpy(), cmap="Blues", norm=PowerNorm(gamma=0.5))
ax[0].set_title("NN Learned Probabilities")

ax[1].imshow(bigram_cond_probs, cmap="Blues", norm=PowerNorm(gamma=0.5))
ax[1].set_title("Bigram Probabilities")

for i in [0, 1]:
    ax[i].set_xticks(range(vocab_size))
    ax[i].set_xticklabels(encoder.vocab)
    ax[i].set_yticks(range(vocab_size))
    ax[i].set_yticklabels(encoder.vocab)

As suggested by the high cosine similarity values, the probability distributions look very similar.

### Exploring the Effects of Regularization

In [ ]:
def run_gradient_descent_with_regularization(
    l2_reg: float,
    learning_rate: float = 50,
    epochs: int = 200,
) -> list[float]:
    # Initialize with standard normal weights
    W = torch.randn((vocab_size, vocab_size), generator=g, requires_grad=True)

    # Training the neural network using gradient descent
    loss_values = []

    for i in range(epochs + 1):
        # Use torch's cross-entropy function this time
        logits = X @ W
        entropy_loss = F.cross_entropy(logits, y, reduction="none")
        weighted_loss = (entropy_loss * sample_weights).sum()

        # Add L2 regularization
        loss = weighted_loss + l2_reg * W.pow(2).mean()
        loss_values.append(loss.item())

        if i % 20 == 0:
            print(f"{i=} | loss={loss.item():.3f}")

        # Backpropagation
        W.grad = None
        loss.backward()

        # Weight update, adjusted by the learning rate
        with torch.no_grad():
            W -= learning_rate * W.grad

    return W, loss_values


def plot_learning_curve(loss_values: list[float], title: str):
    _, ax = plt.subplots(figsize=(8, 4))

    ax.plot(np.arange(len(loss_values)), loss_values)
    ax.axhline(
        bigram_nll,
        color="red",
        linestyle="--",
        label="Computed bigram model",
    )
    ax.axhline(
        uniform_nll,
        color="purple",
        linestyle="--",
        label="Uniform bigram model",
    )
    ax.set_xlabel("epoch")
    ax.set_ylabel("Loss value")
    ax.set_title(title)
    ax.legend()

    return ax

In [ ]:
_, loss_values = run_gradient_descent_with_regularization(l2_reg=0.001)

plot_learning_curve(
    loss_values, title="Loss value during neural net training with mild L2 regularization"
);

A small amount of regularization has minimal impact on the final loss value.

In [ ]:
_, loss_values = run_gradient_descent_with_regularization(l2_reg=10)

plot_learning_curve(
    loss_values, title="Loss value during neural net training with higher L2 regularization"
);

Higher L2 regularization causes the model to underfit the data. Additionally, the learning rate seems to be too high for the model to converge effectively in this scenario.

In [ ]:
W, loss_values = run_gradient_descent_with_regularization(l2_reg=10000, learning_rate=0.01)

plot_learning_curve(
    loss_values, title="Loss value during neural net training with heavy L2 regularization"
);


Extreme regularization and a lower learning rate cause the model weights to converge to nearly zero, making it equivalent to the uniform bigram model.

In [ ]:
torch.allclose(W, torch.zeros_like(W), atol=1e-2)

In [ ]:
bool(np.isclose(loss_values[-1], uniform_nll, atol=1e-2))

### Conclusion

Models that reduce names to sequences of bigrams cannot capture the full complexity of the data, as revealed by the poor quality of the generated names.

Next, let's try building n-gram models to evaluate the improvement with more memory.